In [1]:
from dotenv import load_dotenv
import os

load_dotenv() 

True

In [2]:
!pip install -q langchain-openai langchain-core requests


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a + b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))

7


In [6]:
multiply.name

'multiply'

In [7]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
# tool binding

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)


In [11]:
llm.invoke('hi')

AIMessage(content='Hi! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fccd4-f628-74d2-9724-1631c461bf55-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 268, 'total_tokens': 270, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 259}})

In [12]:
llm_with_tools = llm.bind_tools([multiply])

In [13]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fccd5-1c44-7050-967c-570289faba6e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 16, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [14]:
query = HumanMessage('can you multiply 3 with 1000')

In [15]:
messages = [query]

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [17]:
result = llm_with_tools.invoke(messages)

In [18]:
messages.append(result)

In [19]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 1000, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'cbb22059-948f-45c6-9b28-818cf5e51344': 'Co0CARFNMg/+MgjIi6TJJhqgC1eKul7L6O8ihLpzXQqwqF1s2f7hvjxqbPyPTeO77/esPArj7+AXKTTGokHgC6CqA8NXb7BbEMDC5pAqjHGloHLxjP1e/Qy/6VE4XaEBdeWmucJ01V19xZcJaWwyOjlLecP9wCGayf7T0Qu7XCT1e3yRZDQvKwK2iZUuns2oaZyezGlfSPm4RodpR6TeByicTh+iVP5QXGfeyCX0dYvn4t4qRWlJVnML7E8TPhWQsg9wsGwoU+Nnge2kxnqmRSzBMlQm9MAEipQo2N1CeLfbBY8O0T2PkcE36nydFdaru+F7i2JiQH5FLmV9kTh5AY8mEmqiUEaGHHUGOK5YKEY='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fccd5-22eb-76e2-b008-799df9ca555f-0', tool_calls=[{'name': 'multiply', 'args': {'b': 1000, 'a': 3}, 'id': 'cbb22059-948f-45c6-9b28-818cf5e51344', 'type': 'tool_call'}], invalid

In [20]:
tool_result = multiply.invoke(result.tool_calls[0])

In [21]:
tool_result

ToolMessage(content='1003', name='multiply', tool_call_id='cbb22059-948f-45c6-9b28-818cf5e51344')

In [22]:
messages.append(tool_result)

In [23]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 1000, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'cbb22059-948f-45c6-9b28-818cf5e51344': 'Co0CARFNMg/+MgjIi6TJJhqgC1eKul7L6O8ihLpzXQqwqF1s2f7hvjxqbPyPTeO77/esPArj7+AXKTTGokHgC6CqA8NXb7BbEMDC5pAqjHGloHLxjP1e/Qy/6VE4XaEBdeWmucJ01V19xZcJaWwyOjlLecP9wCGayf7T0Qu7XCT1e3yRZDQvKwK2iZUuns2oaZyezGlfSPm4RodpR6TeByicTh+iVP5QXGfeyCX0dYvn4t4qRWlJVnML7E8TPhWQsg9wsGwoU+Nnge2kxnqmRSzBMlQm9MAEipQo2N1CeLfbBY8O0T2PkcE36nydFdaru+F7i2JiQH5FLmV9kTh5AY8mEmqiUEaGHHUGOK5YKEY='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fccd5-22eb-76e2-b008-799df9ca555f-0', tool_calls=[{'name': 'multiply', 'args': {'b': 1000, 'a': 3}, 'id': 'cbb22059-948f-45c6-9b28-818cf5e51344', 'type': 'tool_call'}], invalid

In [24]:
llm_with_tools.invoke(messages).content

'The product of 3 and 1000 is 1003.'

In [25]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [26]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [27]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1785801601,
 'time_last_update_utc': 'Tue, 04 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1785888001,
 'time_next_update_utc': 'Wed, 05 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.395}

In [28]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [29]:
# tool binding
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

In [30]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [31]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [32]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [33]:
ai_message = llm_with_tools.invoke(messages)

In [34]:
messages.append(ai_message)

In [35]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'b570eac0-d755-4d40-83f7-36a6da676c60',
  'type': 'tool_call'}]

In [36]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [37]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "INR", "target_currency": "USD"}'}, '__gemini_function_call_thought_signatures__': {'b570eac0-d755-4d40-83f7-36a6da676c60': 'CpIcARFNMg92FS8/JzrRNBAXvrs7It6DMZ2bOpqBpvwCuWFLt9zfmy/N8f/Q+bVIcZYnxfM+DtrGZe5KFqzL2OBmZPppvw35zIiYb9nmNjSqpPF+SUy5YtecbibCNOg1TodGrVtBhrh1y3Jw18+b5hlHP1GFQMM6zjIRGDu7w7keiKXOcgGPNi12L91fPQDdHax3Rjatm+xhviOOFZjOXPkL2JPon0foXyFEdzR9mkggKf7nGnoXFGBY9I6rRMRaG1JI2Oz21TEWvOpBwqQu2DsQpMun23ryGUjUzm5pav8q5EEWGexA/JnfLDgzl96e97gTi/D5TgEtSjn+7XMiPQJaCO3LflgsPNvq6gYVUwUuGP76iCXknC8Y2oMoUrSfSUVak1okS63mcrjzsZ7Q2gUaGxges/rUl8r+LwEl2ghf++y87KcVR0Y9wWhQUuCjXcCCgq2cU2Oytk3LQUCmRGkXjlhOGvhybloxzHEVqYXpfCEmvTvAwI9OuIj9T323bIxqfWiRf5PsO7Qx9Ie7/ifi/1SsRnMiX4xw1vKUJPrslwHC1JMKzyWED4BHp9HR3aVcL

In [38]:
llm_with_tools.invoke(messages).content

''

In [39]:
from langchain.agents import create_agent

# Step 5: Initialize the Agent ---
agent_executor = create_agent(
    model=llm,
    tools=[get_conversion_factor, convert, multiply],
    system_prompt="You must always use the provided tools for any arithmetic calculation. Never compute math yourself — always call the appropriate tool."
)

In [40]:
# --- Step 6: Run the Agent ---
user_query = "can u tell me wht is 87 time 45"

response = agent_executor.invoke({"messages": [{"role": "user", "content": user_query}]})
print(response["messages"][-1].content)

I'm sorry, I made a mistake in the previous calculation. 87 multiplied by 45 is 3915.
